In [4]:
from elasticsearch import Elasticsearch

es = Elasticsearch("http://localhost:9200")

print(es.info())

{'name': 'ahmed-HP-EliteBook-840-G1', 'cluster_name': 'elasticsearch', 'cluster_uuid': 'W4B7ErSDS-G_v4p6lxl0sg', 'version': {'number': '9.3.2', 'build_flavor': 'default', 'build_type': 'tar', 'build_hash': '43a703737aab6baefa748bc7b69e4054926f2b2c', 'build_date': '2026-03-16T13:12:56.143057855Z', 'build_snapshot': False, 'lucene_version': '10.3.2', 'minimum_wire_compatibility_version': '8.19.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'}


In [6]:
template = {
    "index_patterns": ["libra-borrow-*"],
    "template": {
        "settings": {
            "number_of_shards": 1
        },

        "mappings": {
            "properties": {
                "user": {"type": "keyword"},
                "book_title": {"type": "text"},
                "borrow_date": {"type": "date"},
                "return_date": {"type": "date"},
                "status": {"type": "keyword"},
                "penality": {"type": "float"}
            }
        }
    }
}

es.indices.put_index_template(name="borrow_template", body=template)
print("template borrow créé")

template borrow créé


In [7]:
es.indices.create(
    index="libra-borrow-v1",
    body={
        "aliases": {
            "borrows": {}
        }
    }
)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'libra-borrow-v1'})

In [9]:
from elasticsearch.helpers import bulk

docs = [
    {
        "_index": "borrows",
        "_source": {
            "user": "ahmed",
            "book_date": "Python Programming",
            "borrow_date": "2024-03-01",
            "return_date": "2024-03-10",
            "status": "returned",
            "penality": 0
        }
    },
    {
        "_index": "borrows",
        "_source": {
            "user": "sara",
            "book_date": "Machine learning Basics",
            "borrow_date": "2024-03-05",
            "return_date": "2024-03-20",
            "status": "late",
            "penality": 15
        }
    },
    {
        "_index": "borrows",
        "_source": {
            "user": "ali",
            "book_date": "Deep learning guide",
            "borrow_date": "2024-03-10",
            "status": "ongoing"
        }
    }
]

bulk(es, docs)
print("Données borrow insérées")

Données borrow insérées


In [10]:
es.index(index="borrows", document={
    "user": "yasmine",
    "book_title": "AI Revolution",
    "borrow_date": "2024-03-15",
    "status": "ongoing"
})

ObjectApiResponse({'_index': 'libra-borrow-v1', '_id': 'IlF3c50BwD4z24wuhuP3', '_version': 1, 'result': 'created', '_shards': {'total': 2, 'successful': 1, 'failed': 0}, '_seq_no': 3, '_primary_term': 1})

In [13]:
res = es.search(index="borrows", query={"match_all": {}})

for hit in res["hits"]["hits"]:
    #print(hit)
    print(hit["_source"])

{'_index': 'libra-borrow-v1', '_id': 'H1F1c50BwD4z24wuiuPS', '_score': 1.0, '_source': {'user': 'ahmed', 'book_date': 'Python Programming', 'borrow_date': '2024-03-01', 'return_date': '2024-03-10', 'status': 'returned', 'penality': 0}}
{'_index': 'libra-borrow-v1', '_id': 'IFF1c50BwD4z24wuiuPS', '_score': 1.0, '_source': {'user': 'sara', 'book_date': 'Machine learning Basics', 'borrow_date': '2024-03-05', 'return_date': '2024-03-20', 'status': 'late', 'penality': 15}}
{'_index': 'libra-borrow-v1', '_id': 'IVF1c50BwD4z24wuiuPS', '_score': 1.0, '_source': {'user': 'ali', 'book_date': 'Deep learning guide', 'borrow_date': '2024-03-10', 'status': 'ongoing'}}
{'_index': 'libra-borrow-v1', '_id': 'IlF3c50BwD4z24wuhuP3', '_score': 1.0, '_source': {'user': 'yasmine', 'book_title': 'AI Revolution', 'borrow_date': '2024-03-15', 'status': 'ongoing'}}


In [14]:
es.search()

es.update(
    index="borrows",
    id="IlF3c50BwD4z24wuhuP3",
    doc={"status": "returned"}
)

ObjectApiResponse({'_index': 'libra-borrow-v1', '_id': 'IlF3c50BwD4z24wuhuP3', '_version': 2, 'result': 'updated', '_shards': {'total': 2, 'successful': 1, 'failed': 0}, '_seq_no': 4, '_primary_term': 1})

In [15]:
es.delete(index="borrows", id="IlF3c50BwD4z24wuhuP3")

ObjectApiResponse({'_index': 'libra-borrow-v1', '_id': 'IlF3c50BwD4z24wuhuP3', '_version': 3, 'result': 'deleted', '_shards': {'total': 2, 'successful': 1, 'failed': 0}, '_seq_no': 5, '_primary_term': 1})

In [16]:
res = es.search(
    index="borrows",
    query={
        "term": {"status": "late"}
    }
)

print("Retards:")
for hit in res["hits"]["hits"]:
    print(hit["_source"])

Retards:
{'user': 'sara', 'book_date': 'Machine learning Basics', 'borrow_date': '2024-03-05', 'return_date': '2024-03-20', 'status': 'late', 'penality': 15}


In [21]:
res = es.search(
    index="borrows",
    query={
        "term": {"status": "ongoing"}
    }
)

print("Ongoing:")
for hit in res["hits"]["hits"]:
    print(hit["_source"])

Ongoing:
{'user': 'ali', 'book_date': 'Deep learning guide', 'borrow_date': '2024-03-10', 'status': 'ongoing'}


In [25]:
res = es.search(
    index="borrows",
    query={
        "range": {
            "borrow_date": {
                "gte": "2024-03-01",
                "lte": "2024-03-10"
            }
        }
    }
)

for hit in res["hits"]["hits"]:
    print(hit["_source"])

{'user': 'ahmed', 'book_date': 'Python Programming', 'borrow_date': '2024-03-01', 'return_date': '2024-03-10', 'status': 'returned', 'penality': 0}
{'user': 'sara', 'book_date': 'Machine learning Basics', 'borrow_date': '2024-03-05', 'return_date': '2024-03-20', 'status': 'late', 'penality': 15}
{'user': 'ali', 'book_date': 'Deep learning guide', 'borrow_date': '2024-03-10', 'status': 'ongoing'}


In [26]:
res = es.search(
    index="borrows",
    query={
        "bool": {
            "must": [
                {"term": {"user": "sara"}},
                {"term": {"status": "late"}}
            ]
        }
    }
)

for hit in res["hits"]["hits"]:
    print(hit["_source"])

{'user': 'sara', 'book_date': 'Machine learning Basics', 'borrow_date': '2024-03-05', 'return_date': '2024-03-20', 'status': 'late', 'penality': 15}


In [27]:
res = es.search(
    index="borrows",
    query={"match_all": {}},
    sort=[{"borrow_date": "desc"}],
    size=2
)

for hit in res["hits"]["hits"]:
    print(hit["_source"])

{'user': 'ali', 'book_date': 'Deep learning guide', 'borrow_date': '2024-03-10', 'status': 'ongoing'}
{'user': 'sara', 'book_date': 'Machine learning Basics', 'borrow_date': '2024-03-05', 'return_date': '2024-03-20', 'status': 'late', 'penality': 15}


In [30]:
res = es.search(
    index="borrows",
    size=0,
    aggs={
        "by_user": {
            "terms": {"field": "user"}
        }
    }
)

print(res["aggregations"])

{'by_user': {'doc_count_error_upper_bound': 0, 'sum_other_doc_count': 0, 'buckets': [{'key': 'ahmed', 'doc_count': 1}, {'key': 'ali', 'doc_count': 1}, {'key': 'sara', 'doc_count': 1}]}}


In [32]:
res = es.search(
    index="borrows",
    size=0,
    aggs={
        "total_penality": {
            "sum": {"field": "penality"}
        }
    }
)

print(res["aggregations"])

{'total_penality': {'value': 15.0}}


In [33]:
res = es.search(
    index="borrows",
    size=0,
    aggs={
        "users_penality": {
            "terms": {"field": "user"},
            "aggs": {
                "total_penality": {
                    "sum": {"field": "penality"}
                }
            }
        }
    }
)

print(res["aggregations"])

{'users_penality': {'doc_count_error_upper_bound': 0, 'sum_other_doc_count': 0, 'buckets': [{'key': 'ahmed', 'doc_count': 1, 'total_penality': {'value': 0.0}}, {'key': 'ali', 'doc_count': 1, 'total_penality': {'value': 0.0}}, {'key': 'sara', 'doc_count': 1, 'total_penality': {'value': 15.0}}]}}


In [34]:
es.indices.create(index="libra-borrow-v2")

es.reindex(body = {
    "source": {"index": "libra-borrow-v1"},
    "dest": {"index": "libra-borrow-v2"}
})

es.indices.update_aliases(body={
    "actions": [
        {"remove": {"index": "libra-borrow-v1", "alias": "borrows"}},
        {"add": {"index": "libra-borrow-v2", "alias": "borrows"}}
    ]
})

ObjectApiResponse({'acknowledged': True, 'errors': False})